## Agente Auditor — Agente A
### Auditoría Semántica de Decisiones en Aseguradora

Este notebook documenta y ejecuta el flujo completo del Agente Auditor.  
Evalúa las decisiones de Agente B usando `sentence-transformers` y controles de negocio definidos en `reglas.json`.

---
**Flujo general:**
1. Cargar casos y reglas
2. Generar embeddings semánticos
3. Calcular Índice de Fidelidad (IF)
4. Evaluar controles de negocio
5. Emitir diagnóstico por caso

### 0. Instalación de dependencias
Ejecutar solo si no están instaladas.

In [ ]:
pip install sentence-transformers scikit-learn numpy python-dotenv

### 1. Importanciones y designación de Rutas

In [ ]:
import json
import re
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Rutas relativas al notebook
BASE_DIR    = Path(".")   
CASOS_PATH  = BASE_DIR / "data"   / "casos.json"
REGLAS_PATH = BASE_DIR / "config" / "reglas.json"

MODELO_NOMBRE = "paraphrase-multilingual-MiniLM-L12-v2"

print("Importaciones OK")

### 2. Carga de Archivos
Se lee `casos.json` con los casos de Agente B y `reglas.json` con los controles de auditoría.

In [ ]:
def cargar_json(ruta: Path, nombre: str):
    with open(ruta, "r", encoding="utf-8") as f:
        return json.load(f)

casos  = cargar_json(CASOS_PATH,  "casos.json")
reglas = cargar_json(REGLAS_PATH, "reglas.json")

print(f"Casos cargados    : {len(casos)}")
print(f"Controles activos : {len(reglas['controles'])}")
print(f"Controles         : {list(reglas['controles'].keys())}")

#### Inspección preliminar de los casos

In [ ]:
for caso in casos:
    print(f"\n{'='*60}")
    print(f"Caso {caso['id_caso']}")
    print(f"Contexto RAG : {caso['contexto_rag']}")
    print(f"Respuesta B  : {caso['respuesta_agent_b']}")

### 3. Modelo Semántico
Se carga `paraphrase-multilingual-MiniLM-L12-v2` - modelo multilingüe para convertir texto en vectores númericos. Permitiendo 
comparar semanticamente dos textos aunque usen palabras distintas.

In [ ]:
modelo = SentenceTransformer(MODELO_NOMBRE)
print(f"Modelo cargado: {MODELO_NOMBRE}")

### 4. Motor Semántico
Funciones que calculan la similitud entre textos usando similitud coseno sobre los embeddings.

In [ ]:
def extraer_montos(texto: str, patron: str) -> list[float]:
    """Extrae montos USD del texto usando el regex del reglas.json"""
    matches = re.findall(patron, texto)
    montos = []
    for m in matches:
        try:
            montos.append(float(m.replace(",", "")))
        except ValueError:
            continue
    return montos

def similitud(embedding_a: np.ndarray, embedding_b: np.ndarray) -> float:
    return float(cosine_similarity([embedding_a], [embedding_b])[0][0])


def similitud_texto_vs_frases(embedding_texto: np.ndarray, frases: list, modelo: SentenceTransformer) -> float:
    if not frases:
        return 0.0
    embeddings_frases = modelo.encode(frases)
    scores = [similitud(embedding_texto, emb) for emb in embeddings_frases]
    return round(max(scores), 4)


def calcular_if_dinamico(caso: dict, reglas: dict, modelo, tau: float = 0.1) -> float:
    contexto  = caso["contexto_rag"]
    respuesta = caso["respuesta_agent_b"]
    emb_respuesta = modelo.encode(respuesta)

    scores = []

    for nombre, ctrl in reglas["controles"].items():
        keywords = ctrl.get("keywords_activacion", [])
        if not any(kw.lower() in contexto.lower() for kw in keywords):
            continue

        tipo = ctrl.get("tipo", "")

        if tipo == "comparacion_numerica":
            patron = reglas["extraccion"]["montos_usd"]["regex"]
            m_ctx = extraer_montos(contexto, patron)
            m_res = extraer_montos(respuesta, patron)
            if m_ctx and m_res:
                limite   = max(m_ctx)
                aprobado = max(m_res)
                # Score gradual — qué tan lejos está del límite
                ratio = aprobado / limite
                if ratio <= 1.0:
                    scores.append(round(1.0 - (ratio * 0.2), 4))  # cerca del límite = score alto pero no 1.0
                else:
                    scores.append(round(max(0.0, 1.0 - ratio + 0.5), 4))  # superó = score bajo gradual

        elif tipo == "deteccion_urgencia":
            keywords_indebidas = ctrl.get("keywords_aprobacion_indebida", [])
            # Contar cuántas keywords indebidas aparecen — más keywords = peor score
            matches = sum(1 for kw in keywords_indebidas if kw.lower() in respuesta.lower())
            total   = len(keywords_indebidas) if keywords_indebidas else 1
            scores.append(round(1.0 - (matches / total), 4))

        elif tipo == "deteccion_derivacion":
            keywords_correctas = ctrl.get("keywords_accion_correcta", [])
            keywords_indebidas = ctrl.get("keywords_aprobacion_indebida", [])
            match_correctas = sum(1 for kw in keywords_correctas if kw.lower() in respuesta.lower())
            match_indebidas = sum(1 for kw in keywords_indebidas if kw.lower() in respuesta.lower())
            total_c = len(keywords_correctas) if keywords_correctas else 1
            total_i = len(keywords_indebidas) if keywords_indebidas else 1
            score_c = match_correctas / total_c
            score_i = match_indebidas / total_i
            scores.append(round(score_c - score_i + 0.5, 4))  # centrado en 0.5

    if not scores:
        return round(similitud(modelo.encode(contexto), modelo.encode(respuesta)), 4)

    return round(min(1.0, max(0.0, sum(scores) / len(scores))), 4)


def clasificar_if(score: float, umbrales: dict) -> str:
    if score >= umbrales["CONFORME"]:
        return "CONFORME"
    return "NO_CONFORME"


print("Motor semántico listo")

### Demo del Indicador de Fidelidad Analítica
Validar el puntaje obtenido para cada caso antes de aplicar los controles.

In [ ]:
umbrales_if = reglas["umbrales"]["if"]

print(f"{'Caso':<8} {'IF Score':<12} {'Categoría'}")
print("-" * 40)

for caso in casos:
    score = calcular_if_dinamico(caso, reglas, modelo)
    cat   = clasificar_if(score, umbrales_if)
    print(f"Caso {caso['id_caso']:<4} {score:<12} {cat}")

### 5. Extracción numerica
Se extraen valores monetarios

In [ ]:
patron_monto = reglas["extraccion"]["montos_usd"]["regex"]

for caso in [casos[0], casos[2]]:
    m_ctx = extraer_montos(caso["contexto_rag"],      patron_monto)
    m_res = extraer_montos(caso["respuesta_agent_b"], patron_monto)
    print(f"Caso {caso['id_caso']} — Contexto: {m_ctx} | Respuesta: {m_res}")

### 6. Controles de negocio
Cada control se evalúa en dos pasos:
1. **¿Aplica?** — El contexto RAG contiene alguna `keyword_activacion` definida.
2. **¿Qué hizo Agente B?** — En base al tipo de control realiza validaciones sobre la respuesta del Agente B.

In [ ]:
def evaluar_control(nombre, ctrl, contexto, respuesta, embedding_contexto, embedding_respuesta, reglas, modelo):   
    # Ancla de Prioridad keyworods sobre score semantico
    keywords_activacion = ctrl.get("keywords_activacion", [])
    activado_x_keyword = any(kw.lower() in contexto.lower() for kw in keywords_activacion)

    if not activado_x_keyword:
        return True, f"Control '{nombre}' no aplica a este caso."

    tipo = ctrl.get("tipo", "")

    if tipo == "deteccion_urgencia":
        # Si hay señal de urgencia en el contexto, Agente B NO debe aprobar
        keywords_indebidas = ctrl.get("keywords_aprobacion_indebida", [])
        aprobo_indebidamente = any(kw.lower() in respuesta.lower() for kw in keywords_indebidas)

        if aprobo_indebidamente:
            return False, ctrl["diagnostico_falla"]
        return True, "Agente B respetó la señal de bloqueo urgente."

    elif tipo == "comparacion_numerica":
        patron_monto = reglas["extraccion"]["montos_usd"]["regex"]
        patron_edad  = reglas["extraccion"]["edad"]["regex"]

        montos_ctx = extraer_montos(contexto, patron_monto)
        montos_res = extraer_montos(respuesta, patron_monto)

        # Comparar montos si existen
        if montos_ctx and montos_res:
            limite   = max(montos_ctx)
            aprobado = max(montos_res)
            if aprobado > limite:
                return False, (
                    f"Monto aprobado (${aprobado:,.0f} USD) supera el límite "
                    f"(${limite:,.0f} USD). {ctrl['diagnostico_falla']}"
                )

        # Comparar edades si existen
        edad_ctx = re.search(patron_edad, contexto, re.IGNORECASE)
        edad_res = re.search(r'\b(\d{2})\s*años?', respuesta, re.IGNORECASE)

        if edad_ctx and edad_res and montos_ctx and montos_res:
            edad_limite  = int(edad_ctx.group(1))
            edad_asegurado = int(edad_res.group(1))
            limite   = max(montos_ctx)
            aprobado = max(montos_res)

            if edad_asegurado >= edad_limite and aprobado > limite:
                return False, (
                    f"Asegurado de {edad_asegurado} años supera el límite de edad "
                    f"con monto ${aprobado:,.0f} USD > ${limite:,.0f} USD. "
                    f"{ctrl['diagnostico_falla']}"
                )

        return True, "Agente B respetó los límites numéricos del contexto."

    elif tipo == "deteccion_derivacion":
        keywords_correctas = ctrl.get("keywords_accion_correcta", [])
        keywords_indebidas = ctrl.get("keywords_aprobacion_indebida", [])

        derivo_correctamente = any(kw.lower() in respuesta.lower() for kw in keywords_correctas)
        aprobo_indebidamente = any(kw.lower() in respuesta.lower() for kw in keywords_indebidas)

        if derivo_correctamente:
            return True, "Agente B derivó correctamente el caso a revisión manual."

        if aprobo_indebidamente:
            return False, ctrl["diagnostico_falla"]

        return True, "Agente B no procesó automáticamente el caso."

    
    return True, f"Control '{nombre}' evaluado sin tipo definido."


print("Controles de negocio listos")

 ### 7. Motor de Diagnostico
 Orquestación del flujo completo del desarrollo: crear embeddings, calcular IF, evaluar controles y definir estado final

In [ ]:
def diagnosticar_caso(caso, reglas, modelo):
    # Auditar caso — crear embeddings, calcular IF, evaluar controles y definir estado final

    contexto  = caso["contexto_rag"]
    respuesta = caso["respuesta_agent_b"]
    id_caso   = caso["id_caso"]

    # Embeddings una sola vez por caso
    embedding_contexto  = modelo.encode(contexto)
    embedding_respuesta = modelo.encode(respuesta)

    if_score    = calcular_if_dinamico(caso, reglas, modelo)
    if_categoria = clasificar_if(if_score, reglas["umbrales"]["if"])

    # Controles en orden de prioridad sugerido
    orden_controles = [
    "control_bloqueo_urgente",
    "control_limite_numerico",
    "control_derivacion",
    ]

    controles_fallidos = []
    razones            = []

    for nombre in orden_controles:
        ctrl = reglas["controles"][nombre]
        aprobado, razon = evaluar_control(
            nombre, ctrl, contexto, respuesta,
            embedding_contexto, embedding_respuesta,
            reglas, modelo
        )
        if not aprobado:
            controles_fallidos.append(nombre)
            razones.append(razon)

    # Resultado
    controles_criticos = [
        n for n in controles_fallidos
        if reglas["controles"][n].get("severidad") == "CRITICA"
    ]

    if controles_criticos:
        estado = "BLOQUEADO"
    elif controles_fallidos:
        estado = "RECHAZADO"
    else:
        estado = "APROBADO"

    diagnostico = " | ".join(razones) if razones else reglas["estados_posibles"][estado]

    return {
        "id_caso":            id_caso,
        "estado":             estado,
        "if_score":           if_score,
        "if_categoria":       if_categoria,
        "diagnostico":        diagnostico,
        "controles_fallidos": controles_fallidos,
    }


print("Motor de diagnóstico listo")

### 8. Ejecución de Auditoria Completa
Se procesan los 4 casos propuestos y se genera diagnostico resultado en la estructura solicitada.

In [ ]:
SEP = "─" * 60
resultados = []

for caso in casos:
    resultado = diagnosticar_caso(caso, reglas, modelo)
    resultados.append(resultado)

    print(SEP)
    print(f"Caso {resultado['id_caso']}: {resultado['estado']}")
    print(f"- Índice de Fidelidad Analítica: {resultado['if_score']} ({resultado['if_categoria']})")
    print(f"- Diagnóstico/Razón: {resultado['diagnostico']}")

print(SEP)